In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import pytz
dbutils.widgets.removeAll()
# Parámetros del Job
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# Configuración Hora Perú
peru_tz = pytz.timezone('America/Lima')
ts_peru = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

df_bronze = spark.table(f"{catalog}.bronze.transacciones_bronze").filter(col("process_date") == process_date)

df_silver = df_bronze.select(
    col("id_cliente").cast("bigint"),
    abs(hash(col("id_cliente"), col("indice_tarjeta"))).alias("id_tarjeta"),
    to_timestamp(
        concat(col("ano"), lit("-"), col("mes"), lit("-"), col("dia"), lit(" "), 
        substring(col("hora"), -8, 8)), # Esto limpia el '2026-03-22 06:21:00' -> '06:21:00'
        "yyyy-M-d HH:mm:ss"
    ).alias("fecha_completa"),
    # Limpieza crítica del monto
    regexp_replace(col("monto"), r"[\$,]", "").cast("double").alias("monto"),
    col("codigo_mcc").cast("string").alias("categoria_mcc"),
    date_format(to_date(concat(col("ano"), lit("-"), col("mes"), lit("-"), col("dia")), "yyyy-M-d"), "EEEE").alias("dia_semana"),
    when(hour(to_timestamp(col("hora"), "HH:mm")).between(22, 23) | hour(to_timestamp(col("hora"), "HH:mm")).between(0, 4), True).otherwise(False).alias("hora_pico"),
    (regexp_replace(col("monto"), r"[\$,]", "").cast("double") / 100).alias("monto_relativo"),
    when(col("es_fraude") == "SÍ", "1").otherwise("0").alias("es_fraude"),
    current_timestamp().alias("ingestion_timestamp"),
    lit(process_date).alias("process_date")
)

target = f"{catalog}.silver.transacciones_silver"
dt = DeltaTable.forName(spark, target)
# Merge por cliente y timestamp exacto para evitar duplicidad de movimientos
dt.alias("t").merge(df_silver.alias("s"), "t.id_cliente = s.id_cliente AND t.fecha_completa = s.fecha_completa") \
  .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# 1. Consultamos la tabla física final en el catálogo
df_final_silver = spark.table(target)

# 2. Obtenemos el conteo total acumulado en Silver
total_registros_silver = df_final_silver.count()

# 3. Obtenemos cuántos registros pertenecen al proceso de hoy (opcional)
registros_hoy = df_final_silver.filter(col("process_date") == process_date).count()

print("═" * 70)
print(f" 📊 REPORTE DE TABLA FÍSICA: {target.upper()} ".center(70, "═"))
print("═" * 70)
print(f"{'Catálogo.Esquema.Tabla':<30} : {target}")
print(f"{'Total registros acumulados':<30} : {total_registros_silver:,}")
print(f"{'Registros cargados/actualizados hoy':<30} : {registros_hoy:,}")
print(f"{'Última sincronización (Perú)':<30} : {ts_peru}")
print("═" * 70)

# 4. Lectura de los 3 registros más recientes de la tabla Silver
print(f"\n🔍 VISTA PREVIA DE LA TABLA SILVER (Top 3 recientes):")

# Ordenamos por timestamp para ver lo que acaba de entrar
display(
    df_final_silver
    .orderBy(col("ingestion_timestamp").desc())
    .limit(3)
)